# Partie C — Embeddings statiques et réseaux profonds

Tokenisation en mots sur le texte normalisé. Le vocabulaire est construit sur train (fréquence ≥ 2). `clean_str` d'AraVec ne sert qu'à la recherche dans son vocabulaire.

C1, C2 et C3 partagent le même budget. L'arrêt anticipé suit le F1 sarcastique de val au seuil 0.5. Le fichier de test officiel n'est pas scoré ; il sert au taux d'OOV.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_and_split
from evaluate import append_results, build_part_c_rows
from train import DEFAULT_SEED, set_seed
from train.deep import TrainConfig, attention_frame, collect_part_c, prepare_part_c

pd.set_option("display.max_colwidth", 160)
set_seed(DEFAULT_SEED)

LIMIT = None  # entier : essai court, results.csv n'est pas modifié
SKIP_ARAVEC = False
CHAR_CNN = False
FOCAL = False

bundle = load_and_split()
config = TrainConfig(seed=DEFAULT_SEED)
print(
    f"budget : lr {config.lr:g}, batch {config.batch_size}, "
    f"{config.max_epochs} époques, patience {config.patience}, hidden {config.hidden}"
)

## Vocabulaire et OOV

La normalisation légère retire l'artefact de guillemets CSV et garde les emojis, les rires, les allongements et la ponctuation. Un type entre dans le vocabulaire s'il apparaît au moins deux fois dans train. Le taux d'OOV est la part des tokens absents de ce vocabulaire.

In [ ]:
prepared = prepare_part_c(bundle["train"], bundle["val"], bundle["test"], limit=LIMIT)
print(f"vocabulaire : {len(prepared.vocab)} types (pad et unk compris)")
print(f"train {len(prepared.train)} — val {len(prepared.val)}")
display(prepared.oov)

## Architectures

- C1. BiLSTM, max-pooling (le dernier état caché est disponible via `TrainConfig.pool = "last"`).
- C2. BiLSTM + attention additive : αᵢ = softmax(vᵀ tanh(W hᵢ)), c = Σ αᵢ hᵢ.
- C3. Mini-Transformer : 2 couches, d_model = 128, 4 têtes, encodage sinusoïdal, masque de padding. L'ablation retire l'encodage positionnel.
- Embeddings : aléatoires, puis AraVec Twitter unigrammes CBOW (100 dim.) figé puis dégelé si `data/aravec/full_uni_cbow_100_twitter.mdl` est présent.

Le déséquilibre est comparé ensuite sur le meilleur modèle de C1–C3 : perte pondérée, sur-échantillonnage de la classe sarcastique dans train, perte standard avec seuil optimisé sur val.

In [ ]:
report = collect_part_c(
    prepared,
    config,
    skip_aravec=SKIP_ARAVEC,
    char_cnn=CHAR_CNN,
    focal=FOCAL,
)
print("AraVec chargé :", report.aravec_loaded)
if report.coverage is not None:
    display(pd.DataFrame([report.coverage]))
print("meilleur C1–C3 :", report.best_model)

## Journal

Les colonnes de test restent vides. Les prédictions de test sont réservées à la partie F.

In [ ]:
y_val = prepared.val["sarcasm"].astype(bool)
rows = build_part_c_rows(y_val, report.runs, seed=DEFAULT_SEED)
display(pd.DataFrame(rows))

if LIMIT is None:
    saved = append_results(rows, ROOT / "results" / "results.csv")
    display(saved)
    print("Prédictions de test non écrites : elles sont réservées à la partie F.")
else:
    print("LIMIT est actif : results.csv n'est pas modifié.")

## Attention (C2)

Poids α du meilleur BiLSTM attentionnel, sur les premiers tweets sarcastiques de val. La question Q7 demande s'ils se concentrent sur les emojis, les rires, les allongements et la ponctuation.

In [ ]:
display(attention_frame(report.attention))